## Using Pydantic model for structured LLM output. 

In the previous lession, you implemented retry machanisms to handle validation error, which mimics what some structured output framework are doing bechind the scenes when they handle validation for you. 

In this lesson you'll experiment with passing you pydantic model directly in your API call using different framework and LLM providers 


By the end of this lesson you'll able to 

- Use pydantic model directly in your API calls to LLMs 
- Reliably recieve a properly structured response using a varity of different framwork and LLM providers 

## Import all the require labraries and set up your environment 

In [2]:
# Import packages 
from pydantic import BaseModel, Field, EmailStr, ValidationError
from typing import List, Literal, Optional
from datetime import date
from dotenv import load_dotenv
import json 
from langchain_groq import ChatGroq
import os 
from groq import Groq

## Define your Pydantic models for user input and LLM output

In [3]:
# Define the UserInput model for customer support queries
# Define the UserInput model for customer support queries
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(
        None,
        description="5-digit order number (cannot start with 0)",
        ge=10000,
        le=99999
    )
    purchase_date: Optional[date] = None

# Define the CustomerQuery model that inherits from UserInput
class CustomerQuery(UserInput):
    priority: str = Field(
        ..., description="Priority level: low, medium, high"
    )
    category: Literal[
        'refund_request', 'information_request', 'other'
    ] = Field(..., description="Query category")
    is_complaint: bool = Field(
        ..., description="Whether this is a complaint"
    )
    tags: List[str] = Field(..., description="Relevant keyword tags")

## provide sample input and validate it using your model 

In [4]:
# Define your input data as a JSON string
user_input_json = '''{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.",
    "order_number": 12345,
    "purchase_date": "2025-12-31"
}'''

In [5]:
# Validate the user_input_json by creating a UserInput instance
user_input = UserInput.model_validate_json(user_input_json)

## Build a prompt and call the GROQ API with the instructor package for structured output

In [6]:
prompt = (
    f"Analyze the following customer query {user_input} "
    f"and provide a structured response."
)

In [12]:
from groq import Groq
import instructor

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [18]:
# Initialize with API key
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Enable instructor patches for Groq client
client = instructor.from_provider("groq/llama-3.1-8b-instant")

In [20]:
# Create structured output
response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt},
    ],
    response_model=CustomerQuery,
)

print(response)

name='Joe User' email='joe.user@example.com' query='I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.' order_id=99998 purchase_date=datetime.date(2025, 12, 31) priority='high' category='refund_request' is_complaint=False tags=['customer_service', 'exchange_policy', 'defective_product', 'replacement_request']


In [30]:
# Inspect the returned structured data
print(type(response))
print(response.model_dump_json(indent=2))

<class '__main__.CustomerQuery'>
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.",
  "order_id": 99998,
  "purchase_date": "2025-12-31",
  "priority": "high",
  "category": "refund_request",
  "is_complaint": false,
  "tags": [
    "customer_service",
    "exchange_policy",
    "defective_product",
    "replacement_request"
  ]
}


In [31]:
# Try out the Pydantic AI package for defining an agent and getting a structured response
from pydantic_ai import Agent
import nest_asyncio
nest_asyncio.apply()

agent = Agent(
    model="groq:llama-3.1-8b-instant",
    output_type=CustomerQuery,
)

response = agent.run_sync(prompt)
print(response)            # You should get a structured CustomerQuery result (or error)
print(type(response.output))  # e.g. <class '__main__.CustomerQuery'>
print(response.output.dict())


AgentRunResult(output=CustomerQuery(name='Joe User', email='joe.user@example.com', query='I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.', order_id=None, purchase_date=datetime.date(2025, 12, 31), priority='high', category='refund_request', is_complaint=True, tags=['defective product', 'replacement', 'refund request']))
<class '__main__.CustomerQuery'>
{'name': 'Joe User', 'email': 'joe.user@example.com', 'query': 'I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.', 'order_id': None, 'purchase_date': datetime.date(2025, 12, 31), 'priority': 'high', 'category': 'refund_request', 'is_complaint': True, 'tags': ['defective product', 'replacement', 'refund request']}


/var/folders/wg/8rqwg0nd7995t2tzgq40dsph0000gn/T/ipykernel_50896/3141933459.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(response.output.dict())


In [32]:
# Print out the repsonse type and content
print(type(response.output))
print(response.output.model_dump_json(indent=2))

<class '__main__.CustomerQuery'>
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.",
  "order_id": null,
  "purchase_date": "2025-12-31",
  "priority": "high",
  "category": "refund_request",
  "is_complaint": true,
  "tags": [
    "defective product",
    "replacement",
    "refund request"
  ]
}
